In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()

while PROJECT_ROOT.name != "Fraud-detection-ML-V2" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

TRAIN_PATH = PROCESSED_DATA_DIR / "cleaned_fraud_train.csv"
TEST_PATH = PROCESSED_DATA_DIR / "cleaned_fraud_test.csv"

if not TRAIN_PATH.exists():
    raise FileNotFoundError(
        f"Training file not found:\n{TRAIN_PATH}"
    )

if not TEST_PATH.exists():
    raise FileNotFoundError(
        f"Test file not found:\n{TEST_PATH}"
    )

train_data = pd.read_csv(TRAIN_PATH)
test_data = pd.read_csv(TEST_PATH)

print("========== PHASE 3 DATA LOADED ==========")
print("Training shape:", train_data.shape)
print("Test shape:", test_data.shape)
print("Training file:", TRAIN_PATH.name)
print("Test file:", TEST_PATH.name)
print("=========================================")

========== PHASE 3 DATA LOADED ==========
Training shape: (1296675, 25)
Test shape: (555719, 25)
Training file: cleaned_fraud_train.csv
Test file: cleaned_fraud_test.csv


In [2]:
required_columns = [
    "transaction_id",
    "user_id",
    "merchant",
    "category",
    "amt",
    "lat",
    "long",
    "merch_lat",
    "merch_long",
    "trans_date_trans_time",
    "is_fraud"
]

missing_train = [
    column
    for column in required_columns
    if column not in train_data.columns
]

missing_test = [
    column
    for column in required_columns
    if column not in test_data.columns
]

if missing_train:
    raise ValueError(
        f"Missing training columns: {missing_train}"
    )

if missing_test:
    raise ValueError(
        f"Missing test columns: {missing_test}"
    )

print("========== FEATURE INPUT VALIDATION ==========")
print(
    "Training columns valid:",
    len(missing_train) == 0
)
print(
    "Test columns valid:",
    len(missing_test) == 0
)
print("===============================================")

========== FEATURE INPUT VALIDATION ==========
Training columns valid: True
Test columns valid: True


In [3]:
train_data["_dataset_split"] = "train"
test_data["_dataset_split"] = "test"

df = pd.concat(
    [train_data, test_data],
    ignore_index=True
)

df["trans_date_trans_time"] = pd.to_datetime(
    df["trans_date_trans_time"],
    errors="coerce"
)

df["amt"] = pd.to_numeric(
    df["amt"],
    errors="coerce"
)

df["lat"] = pd.to_numeric(
    df["lat"],
    errors="coerce"
)

df["long"] = pd.to_numeric(
    df["long"],
    errors="coerce"
)

df["merch_lat"] = pd.to_numeric(
    df["merch_lat"],
    errors="coerce"
)

df["merch_long"] = pd.to_numeric(
    df["merch_long"],
    errors="coerce"
)

df = df.sort_values(
    [
        "user_id",
        "trans_date_trans_time",
        "transaction_id"
    ]
).reset_index(
    drop=True
)

print("========== CHRONOLOGICAL DATA ==========")
print("Combined rows:", len(df))
print("Training rows:", (df["_dataset_split"] == "train").sum())
print("Test rows:", (df["_dataset_split"] == "test").sum())
print(
    "Chronological order valid:",
    df.groupby("user_id")[
        "trans_date_trans_time"
    ].apply(
        lambda x: x.is_monotonic_increasing
    ).all()
)
print("=========================================")

========== CHRONOLOGICAL DATA ==========
Combined rows: 1852394
Training rows: 1296675
Test rows: 555719
Chronological order valid: True


In [4]:
df["previous_timestamp"] = (
    df.groupby("user_id")[
        "trans_date_trans_time"
    ].shift(1)
)

df["previous_merchant_lat"] = (
    df.groupby("user_id")[
        "merch_lat"
    ].shift(1)
)

df["previous_merchant_long"] = (
    df.groupby("user_id")[
        "merch_long"
    ].shift(1)
)

df["previous_category"] = (
    df.groupby("user_id")[
        "category"
    ].shift(1)
)

df["time_since_last_txn_sec"] = (
    df["trans_date_trans_time"]
    - df["previous_timestamp"]
).dt.total_seconds()

df["time_since_last_txn_sec"] = (
    df["time_since_last_txn_sec"]
    .fillna(0)
    .clip(lower=0)
)

print("========== PREVIOUS TRANSACTION DATA ==========")
print(
    df[
        [
            "user_id",
            "trans_date_trans_time",
            "previous_timestamp",
            "time_since_last_txn_sec"
        ]
    ].head(10)
)
print("================================================")

========== PREVIOUS TRANSACTION DATA ==========
       user_id trans_date_trans_time  previous_timestamp  \
0  60416207185   2019-01-01 12:47:15                 NaT   
1  60416207185   2019-01-02 08:44:57 2019-01-01 12:47:15   
2  60416207185   2019-01-02 08:47:36 2019-01-02 08:44:57   
3  60416207185   2019-01-02 12:38:14 2019-01-02 08:47:36   
4  60416207185   2019-01-02 13:10:46 2019-01-02 12:38:14   
5  60416207185   2019-01-03 13:56:35 2019-01-02 13:10:46   
6  60416207185   2019-01-03 17:05:10 2019-01-03 13:56:35   
7  60416207185   2019-01-04 13:59:55 2019-01-03 17:05:10   
8  60416207185   2019-01-04 21:17:22 2019-01-04 13:59:55   
9  60416207185   2019-01-05 00:42:24 2019-01-04 21:17:22   

   time_since_last_txn_sec  
0                      0.0  
1                  71862.0  
2                    159.0  
3                  13838.0  
4                   1952.0  
5                  89149.0  
6                  11315.0  
7                  75285.0  
8                  26247.0  
9

In [5]:
df["previous_amount_sum"] = (
    df.groupby("user_id")["amt"].cumsum()
    - df["amt"]
)

df["previous_transaction_count"] = (
    df.groupby("user_id").cumcount()
)

df["previous_average_amount"] = np.where(
    df["previous_transaction_count"] > 0,
    df["previous_amount_sum"]
    / df["previous_transaction_count"],
    df["amt"]
)

df["amount_vs_avg_ratio"] = np.where(
    df["previous_transaction_count"] > 0,
    df["amt"]
    / df["previous_average_amount"].replace(0, np.nan),
    1.0
)

df["amount_vs_avg_ratio"] = (
    pd.Series(
        df["amount_vs_avg_ratio"]
    )
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(1.0)
    .clip(
        lower=0
    )
)

print("========== AMOUNT RATIO ==========")
print(
    df[
        [
            "amt",
            "previous_average_amount",
            "amount_vs_avg_ratio"
        ]
    ].head(10)
)
print("==================================")

========== AMOUNT RATIO ==========
      amt  previous_average_amount  amount_vs_avg_ratio
0    7.27                 7.270000             1.000000
1   52.94                 7.270000             7.281981
2   82.08                30.105000             2.726457
3   34.79                47.430000             0.733502
4   27.18                44.270000             0.613960
5    6.87                40.852000             0.168168
6    8.43                35.188333             0.239568
7  117.11                31.365714             3.733695
8   26.74                42.083750             0.635400
9  105.20                40.378889             2.605322


In [6]:
def calculate_rolling_transaction_count(group):
    timestamps = (
        group["trans_date_trans_time"]
        .astype("int64")
        .to_numpy()
    )

    window_seconds = 5 * 60

    left_positions = np.searchsorted(
        timestamps,
        timestamps - window_seconds,
        side="left"
    )

    counts = (
        np.arange(len(group))
        - left_positions
    )

    result = pd.Series(
        counts,
        index=group.index
    )

    return result


df["txn_count_last_5min"] = (
    df.groupby(
        "user_id",
        group_keys=False
    )
    .apply(
        calculate_rolling_transaction_count,
        include_groups=False
    )
    .sort_index()
)

df["txn_count_last_5min"] = (
    df["txn_count_last_5min"]
    .fillna(0)
    .astype(int)
)

print("========== VELOCITY FEATURE ==========")
print(
    df[
        [
            "user_id",
            "trans_date_trans_time",
            "txn_count_last_5min"
        ]
    ].head(15)
)
print("======================================")

========== VELOCITY FEATURE ==========
        user_id trans_date_trans_time  txn_count_last_5min
0   60416207185   2019-01-01 12:47:15                    0
1   60416207185   2019-01-02 08:44:57                    0
2   60416207185   2019-01-02 08:47:36                    0
3   60416207185   2019-01-02 12:38:14                    0
4   60416207185   2019-01-02 13:10:46                    0
5   60416207185   2019-01-03 13:56:35                    0
6   60416207185   2019-01-03 17:05:10                    0
7   60416207185   2019-01-04 13:59:55                    0
8   60416207185   2019-01-04 21:17:22                    0
9   60416207185   2019-01-05 00:42:24                    0
10  60416207185   2019-01-05 21:34:20                    0
11  60416207185   2019-01-06 10:25:49                    0
12  60416207185   2019-01-07 12:58:19                    0
13  60416207185   2019-01-08 08:05:23                    0
14  60416207185   2019-01-08 23:20:22                    0


In [7]:
def haversine_km(
    lat1,
    lon1,
    lat2,
    lon2
):
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(
        np.sqrt(a)
    )

    return 6371.0 * c


df["distance_from_last_location_km"] = haversine_km(
    df["previous_merchant_lat"],
    df["previous_merchant_long"],
    df["merch_lat"],
    df["merch_long"]
)

df["distance_from_last_location_km"] = (
    pd.Series(
        df["distance_from_last_location_km"]
    )
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
    .clip(lower=0)
)

print("========== DISTANCE FEATURE ==========")
print(
    df[
        [
            "previous_merchant_lat",
            "previous_merchant_long",
            "merch_lat",
            "merch_long",
            "distance_from_last_location_km"
        ]
    ].head(10)
)
print("======================================")

========== DISTANCE FEATURE ==========
   previous_merchant_lat  previous_merchant_long  merch_lat  merch_long  \
0                    NaN                     NaN  43.974711 -109.741904   
1              43.974711             -109.741904  42.018766 -109.044172   
2              42.018766             -109.044172  42.961335 -109.157564   
3              42.961335             -109.157564  42.228227 -108.747683   
4              42.228227             -108.747683  43.321745 -108.091143   
5              43.321745             -108.091143  43.477317 -109.467136   
6              43.477317             -109.467136  42.871477 -109.160268   
7              42.871477             -109.160268  43.332599 -108.318444   
8              43.332599             -108.318444  43.598123 -108.977767   
9              43.598123             -108.977767  42.314401 -108.554520   

   distance_from_last_location_km  
0                        0.000000  
1                      224.769219  
2                      105.

In [8]:
df["merchant_category_is_new_for_user"] = (
    df.groupby(
        [
            "user_id",
            "category"
        ]
    )
    .cumcount()
    .eq(0)
    .astype(int)
)

print("========== MERCHANT CATEGORY FEATURE ==========")
print(
    df[
        [
            "user_id",
            "category",
            "merchant_category_is_new_for_user"
        ]
    ].head(15)
)
print("===============================================")

========== MERCHANT CATEGORY FEATURE ==========
        user_id       category  merchant_category_is_new_for_user
0   60416207185       misc_net                                  1
1   60416207185  gas_transport                                  1
2   60416207185  gas_transport                                  0
3   60416207185      kids_pets                                  1
4   60416207185           home                                  1
5   60416207185   shopping_net                                  1
6   60416207185    food_dining                                  1
7   60416207185           home                                  0
8   60416207185  personal_care                                  1
9   60416207185    grocery_pos                                  1
10  60416207185  personal_care                                  0
11  60416207185    grocery_pos                                  0
12  60416207185   shopping_net                                  0
13  60416207185  gas_transpo

In [9]:
ML_FEATURES = [
    "amount",
    "amount_vs_avg_ratio",
    "txn_count_last_5min",
    "time_since_last_txn_sec",
    "distance_from_last_location_km",
    "merchant_category_is_new_for_user"
]

df["amount"] = df["amt"].astype(float)

ml_columns = [
    "transaction_id",
    "user_id",
    "amount",
    "amount_vs_avg_ratio",
    "txn_count_last_5min",
    "time_since_last_txn_sec",
    "distance_from_last_location_km",
    "merchant_category_is_new_for_user",
    "is_fraud",
    "_dataset_split"
]

ml_df = df[ml_columns].copy()

print("========== FINAL ML FEATURES ==========")
print("Number of ML features:", len(ML_FEATURES))
print("ML features:")
print(ML_FEATURES)
print()
print("ML dataset shape:", ml_df.shape)
print("=======================================")

========== FINAL ML FEATURES ==========
Number of ML features: 6
ML features:
['amount', 'amount_vs_avg_ratio', 'txn_count_last_5min', 'time_since_last_txn_sec', 'distance_from_last_location_km', 'merchant_category_is_new_for_user']

ML dataset shape: (1852394, 10)


In [10]:
feature_missing_values = (
    ml_df[ML_FEATURES]
    .isnull()
    .sum()
)

feature_infinite_values = (
    np.isinf(
        ml_df[ML_FEATURES]
        .astype(float)
    )
    .sum()
)

feature_ranges = (
    ml_df[ML_FEATURES]
    .describe()
    .transpose()
)

print("========== FEATURE VALIDATION ==========")

print("Missing values:")
print(feature_missing_values)

print()
print("Infinite values:")
print(feature_infinite_values)

print()
print("Feature statistics:")
print(feature_ranges)

print()
print(
    "All feature values finite:",
    not np.isinf(
        ml_df[ML_FEATURES]
        .astype(float)
    ).any().any()
)

print(
    "All required features present:",
    all(
        feature in ml_df.columns
        for feature in ML_FEATURES
    )
)

print("========================================")

========== FEATURE VALIDATION ==========
Missing values:
amount                               0
amount_vs_avg_ratio                  0
txn_count_last_5min                  0
time_since_last_txn_sec              0
distance_from_last_location_km       0
merchant_category_is_new_for_user    0
dtype: int64

Infinite values:
amount                               0
amount_vs_avg_ratio                  0
txn_count_last_5min                  0
time_since_last_txn_sec              0
distance_from_last_location_km       0
merchant_category_is_new_for_user    0
dtype: int64

Feature statistics:
                                       count          mean           std  \
amount                             1852394.0     70.063567    159.253975   
amount_vs_avg_ratio                1852394.0      1.013238      2.591848   
txn_count_last_5min                1852394.0      0.000024      0.004874   
time_since_last_txn_sec            1852394.0  30922.794614  45322.458627   
distance_from_last_location_km

In [11]:
leakage_checks = {
    "previous_timestamp_exists_only_after_first_user_txn": (
        df.groupby("user_id")["previous_timestamp"]
        .apply(
            lambda x:
            x.isna().sum() >= 1
        )
        .all()
    ),
    "time_since_last_txn_nonnegative": (
        (df["time_since_last_txn_sec"] >= 0).all()
    ),
    "rolling_count_nonnegative": (
        (df["txn_count_last_5min"] >= 0).all()
    ),
    "distance_nonnegative": (
        (df["distance_from_last_location_km"] >= 0).all()
    ),
    "category_flag_binary": (
        df["merchant_category_is_new_for_user"]
        .isin([0, 1])
        .all()
    )
}

print("========== LEAKAGE / LOGIC CHECK ==========")

for name, result in leakage_checks.items():
    print(
        f"{name}:",
        result
    )

print()

print(
    "All checks passed:",
    all(leakage_checks.values())
)

print("============================================")

========== LEAKAGE / LOGIC CHECK ==========
previous_timestamp_exists_only_after_first_user_txn: True
time_since_last_txn_nonnegative: True
rolling_count_nonnegative: True
distance_nonnegative: True
category_flag_binary: True

All checks passed: True


In [12]:
ml_train_df = (
    ml_df[
        ml_df["_dataset_split"] == "train"
    ]
    .drop(
        columns=["_dataset_split"]
    )
    .reset_index(
        drop=True
    )
)

ml_test_df = (
    ml_df[
        ml_df["_dataset_split"] == "test"
    ]
    .drop(
        columns=["_dataset_split"]
    )
    .reset_index(
        drop=True
    )
)

print("========== FEATURE DATASET SPLIT ==========")
print("Training feature shape:", ml_train_df.shape)
print("Test feature shape:", ml_test_df.shape)
print()
print(
    "Training fraud count:",
    int(
        (ml_train_df["is_fraud"] == 1).sum()
    )
)
print(
    "Training legitimate count:",
    int(
        (ml_train_df["is_fraud"] == 0).sum()
    )
)
print()
print(
    "Test fraud count:",
    int(
        (ml_test_df["is_fraud"] == 1).sum()
    )
)
print(
    "Test legitimate count:",
    int(
        (ml_test_df["is_fraud"] == 0).sum()
    )
)
print("============================================")

========== FEATURE DATASET SPLIT ==========
Training feature shape: (1296675, 9)
Test feature shape: (555719, 9)

Training fraud count: 7506
Training legitimate count: 1289169

Test fraud count: 2145
Test legitimate count: 553574


In [13]:
TRAIN_FEATURE_PATH = (
    PROCESSED_DATA_DIR
    / "ml_training_features.csv"
)

TEST_FEATURE_PATH = (
    PROCESSED_DATA_DIR
    / "ml_test_features.csv"
)

ml_train_df.to_csv(
    TRAIN_FEATURE_PATH,
    index=False
)

ml_test_df.to_csv(
    TEST_FEATURE_PATH,
    index=False
)

print("========== FEATURE FILES SAVED ==========")
print(
    "Training features:",
    TRAIN_FEATURE_PATH
)

print(
    "Training file exists:",
    TRAIN_FEATURE_PATH.exists()
)

print()

print(
    "Test features:",
    TEST_FEATURE_PATH
)

print(
    "Test file exists:",
    TEST_FEATURE_PATH.exists()
)

print("=========================================")

========== FEATURE FILES SAVED ==========
Training features: C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\data\processed\ml_training_features.csv
Training file exists: True

Test features: C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\data\processed\ml_test_features.csv
Test file exists: True


In [14]:
print()
print("================================================")
print("     STREAMSENTINEL V2 — PHASE 3 SUMMARY")
print("================================================")

print()

print(
    "Training feature rows:",
    len(ml_train_df)
)

print(
    "Test feature rows:",
    len(ml_test_df)
)

print(
    "Number of ML features:",
    len(ML_FEATURES)
)

print(
    "ML features:",
    ML_FEATURES
)

print()

print(
    "Training missing feature values:",
    int(
        ml_train_df[ML_FEATURES]
        .isnull()
        .sum()
        .sum()
    )
)

print(
    "Test missing feature values:",
    int(
        ml_test_df[ML_FEATURES]
        .isnull()
        .sum()
        .sum()
    )
)

print(
    "All leakage checks passed:",
    all(
        leakage_checks.values()
    )
)

print(
    "Training file saved:",
    TRAIN_FEATURE_PATH.exists()
)

print(
    "Test feature file saved:",
    TEST_FEATURE_PATH.exists()
)

print()
print("PHASE 3 BEHAVIORAL FEATURE ENGINEERING: COMPLETE")
print("================================================")


     STREAMSENTINEL V2 — PHASE 3 SUMMARY

Training feature rows: 1296675
Test feature rows: 555719
Number of ML features: 6
ML features: ['amount', 'amount_vs_avg_ratio', 'txn_count_last_5min', 'time_since_last_txn_sec', 'distance_from_last_location_km', 'merchant_category_is_new_for_user']

Training missing feature values: 0
Test missing feature values: 0
All leakage checks passed: True
Training file saved: True
Test feature file saved: True

PHASE 3 BEHAVIORAL FEATURE ENGINEERING: COMPLETE
